In [ ]:
class Brain:

    def __init__(self, targets):

        self.targets = targets
        self.state = "SEARCHING"

        self.locked_target = None
        self.collected = []

        self.max_reach = 1.5

        self.retry_count = 0
        self.max_retries = 2

    # --------------------------------
    # Check reachability
    # --------------------------------
    def reachable(self, position):

        z = position[2]
        return 0 < z < self.max_reach


    # --------------------------------
    # Find closest object
    # --------------------------------
    def closest(self, objects):

        return min(objects, key=lambda x: x["position"][2])


    # --------------------------------
    # Robot attention system
    # --------------------------------
    def focus_target(self, obj):

        self.locked_target = obj["id"]

        return {
            "action": "FOCUS",
            "object": obj["object"],
            "id": obj["id"],
            "position": obj["position"]
        }


    # --------------------------------
    # Main reasoning
    # --------------------------------
    def think(self, world):

        world.decay()

        # If robot already grasping something
        if self.state == "GRASPING":

            print("[BRAIN] Currently grasping object")
            return {"action": "WAIT"}


        # Look for targets
        for target in self.targets:

            if target in self.collected:
                continue

            objects = world.get_by_label(target)

            if not objects:
                continue


            # Select closest candidate
            obj = self.closest(objects)

            print(f"[BRAIN] Target detected: {obj['object']} ID:{obj['id']} dist:{obj['position'][2]:.2f}m")


            # Stability check
            if obj["stable_frames"] < 3:

                print("[BRAIN] Object unstable, focusing")
                return self.focus_target(obj)


            # Reachability check
            if not self.reachable(obj["position"]):

                print("[BRAIN] Object out of reach")
                return {"action": "SEARCH"}


            # Confidence check
            if obj["confidence"] > 0.7:

                print("[BRAIN] Target locked, initiating pick")

                self.locked_target = obj["id"]
                self.state = "GRASPING"

                return {
                    "action": "PICK",
                    "object": obj["object"],
                    "id": obj["id"],
                    "position": obj["position"]
                }


        print("[BRAIN] No targets found, searching")
        return {"action": "SEARCH"}


    # --------------------------------
    # Pick success
    # --------------------------------
    def pick_success(self):

        print("[BRAIN] Pick successful")

        self.collected.append(self.locked_target)

        self.locked_target = None
        self.retry_count = 0

        self.state = "SEARCHING"


    # --------------------------------
    # Pick failure
    # --------------------------------
    def pick_failed(self):

        print("[BRAIN] Pick failed")

        self.retry_count += 1

        if self.retry_count > self.max_retries:

            print("[BRAIN] Max retries reached, abandoning target")

            self.locked_target = None
            self.retry_count = 0
            self.state = "SEARCHING"

        else:

            print("[BRAIN] Retrying grasp")
            self.state = "SEARCHING"